# Chapter 35 — Transfer Learning, Fine-Tuning, and Embeddings

*From Absolute Zero* — companion notebook.

Every block below is the code printed in the chapter, in the same order. Run the cells top to bottom; the output should match the book exactly. If it does not, check `requirements.txt` first, then the errata page.

In [ ]:
!pip -q install -r https://raw.githubusercontent.com/USER/from-absolute-zero/main/requirements.txt  # Colab only; skip locally

## Shared setup

Imports and the objects the blocks below reuse. The chapter prints these once and then continues the same session.

In [ ]:
import numpy as np, warnings; warnings.filterwarnings("ignore")
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from numpy.lib.stride_tricks import sliding_window_view

digits = load_digits()
X_all = digits.images / 16.0
y_all = digits.target

# Source task: digits 0-4. Target task: digits 5-9, a disjoint set of
# classes the source network never saw during pretraining.
src_mask = y_all < 5
tgt_mask = y_all >= 5

Xsrc, ysrc = X_all[src_mask], y_all[src_mask]
Xtgt, ytgt = X_all[tgt_mask], y_all[tgt_mask] - 5     # relabel to 0-4

Xsrc_tr, Xsrc_te, ysrc_tr, ysrc_te = train_test_split(
    Xsrc, ysrc, test_size=0.2, stratify=ysrc, random_state=0)
Xtgt_tr, Xtgt_te, ytgt_tr, ytgt_te = train_test_split(
    Xtgt, ytgt, test_size=0.3, stratify=ytgt, random_state=0)

def softmax(z):
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)

def conv_forward(imgs, filters):
    windows = sliding_window_view(imgs, (3, 3), axis=(1, 2))
    return np.einsum('nijhw,fhw->nfij', windows, filters), windows

def conv_backward(dout, windows, filters):
    dfilters = np.einsum('nfij,nijhw->fhw', dout, windows)
    flipped = filters[:, ::-1, ::-1]
    pad = filters.shape[1] - 1
    dout_p = np.pad(dout, ((0, 0), (0, 0), (pad, pad), (pad, pad)))
    dwindows = sliding_window_view(dout_p, (3, 3), axis=(2, 3))
    dimgs = np.einsum('nfijhw,fhw->nij', dwindows, flipped)
    return dimgs, dfilters

def pool_forward(feats, size=2):
    n, nf, h, w = feats.shape
    r = feats.reshape(n, nf, h // size, size, w // size, size)
    out = r.max(axis=(3, 5))
    mask = (r == out[:, :, :, None, :, None])
    return out, mask

def pool_backward(dout, mask, size=2):
    n, nf, oh, ow = dout.shape
    d = dout[:, :, :, None, :, None] * mask
    return d.reshape(n, nf, oh * size, ow * size)

## The chapter code

### Block 1  (`c1.py`)

In [ ]:
# Pretrain a small CNN to distinguish digits zero through four. The
# filters it learns are Chapter 32's exact architecture, trained on a
# task that has never seen a five, six, seven, eight, or nine.
def train_cnn(Xtr, ytr, Xte, yte, n_classes, seed, epochs=150, n_filters=4):
    r = np.random.default_rng(seed)
    filters = r.normal(0, np.sqrt(2 / 9), (n_filters, 3, 3))
    D_flat = n_filters * 3 * 3
    Wf = r.normal(0, np.sqrt(2 / D_flat), (D_flat, n_classes))
    bf = np.zeros(n_classes)
    Y = np.eye(n_classes)[ytr]
    eta = 0.3
    for epoch in range(epochs):
        conv_out, windows = conv_forward(Xtr, filters)
        relu_out = np.maximum(0, conv_out)
        pool_out, mask = pool_forward(relu_out)
        flat = pool_out.reshape(len(Xtr), -1)
        p = softmax(flat @ Wf + bf)
        dscore = (p - Y) / len(Xtr)
        dWf = flat.T @ dscore
        dbf = dscore.sum(0)
        dflat = dscore @ Wf.T
        dpool = dflat.reshape(pool_out.shape)
        drelu = pool_backward(dpool, mask)
        dconv = drelu * (conv_out > 0)
        _, dfilters = conv_backward(dconv, windows, filters)
        Wf -= eta * dWf; bf -= eta * dbf
        filters -= eta * dfilters
    c_te, _ = conv_forward(Xte, filters)
    p_te, _ = pool_forward(np.maximum(0, c_te))
    pred = softmax(p_te.reshape(len(Xte), -1) @ Wf + bf).argmax(1)
    acc = (pred == yte).mean()
    return filters, Wf, bf, acc

src_filters, src_Wf, src_bf, src_acc = train_cnn(
    Xsrc_tr, ysrc_tr, Xsrc_te, ysrc_te, n_classes=5, seed=35)
print(f"source task (digits 0-4) test accuracy: {src_acc:.4f}")
print(f"learned filters shape: {src_filters.shape}")
print(f"\nthese four filters have seen only zeros, ones, twos, threes,")
print(f"and fours. They have never seen a five, six, seven, eight, or nine.")

### Block 2  (`c2.py`)

In [ ]:
# Freeze the source-trained filters and use them only to extract
# features from the target task's images. Train nothing but a new
# linear head. A random, untrained set of filters is the control: if
# frozen features help only because ANY fixed projection helps, the
# random filters should do just as well.
def extract_features(imgs, filters):
    c, _ = conv_forward(imgs, filters)
    p, _ = pool_forward(np.maximum(0, c))
    return p.reshape(len(imgs), -1)

def train_head(feat_tr, ytr, feat_te, yte, n_classes, seed, epochs=200):
    r = np.random.default_rng(seed)
    D = feat_tr.shape[1]
    W = r.normal(0, np.sqrt(1 / D), (D, n_classes))
    b = np.zeros(n_classes)
    Y = np.eye(n_classes)[ytr]
    eta = 0.5
    for _ in range(epochs):
        p = softmax(feat_tr @ W + b)
        dscore = (p - Y) / len(feat_tr)
        W -= eta * (feat_tr.T @ dscore)
        b -= eta * dscore.sum(0)
    pred = softmax(feat_te @ W + b).argmax(1)
    return (pred == yte).mean()

feat_tr_transfer = extract_features(Xtgt_tr, src_filters)
feat_te_transfer = extract_features(Xtgt_te, src_filters)
acc_transfer = train_head(feat_tr_transfer, ytgt_tr, feat_te_transfer, ytgt_te,
                          n_classes=5, seed=35)

r_rand = np.random.default_rng(999)
random_filters = r_rand.normal(0, np.sqrt(2/9), src_filters.shape)
feat_tr_random = extract_features(Xtgt_tr, random_filters)
feat_te_random = extract_features(Xtgt_te, random_filters)
acc_random = train_head(feat_tr_random, ytgt_tr, feat_te_random, ytgt_te,
                        n_classes=5, seed=35)

print(f"target task (digits 5-9), linear head only, no filter training")
print(f"  frozen SOURCE-TRAINED filters: {acc_transfer:.4f}")
print(f"  frozen RANDOM filters:         {acc_random:.4f}")
print(f"\nsame architecture, same amount of training, only the filters differ.")

### Block 3  (`c3.py`)

In [ ]:
# Transfer learning is expected to earn its keep when target data is
# scarce. Vary how much target training data is available and compare
# frozen transfer against training a small CNN from scratch on that
# same, limited data, at every scale from extreme few-shot upward.
def train_cnn_head_only(feat_tr, ytr, n_classes, seed, epochs=200):
    r = np.random.default_rng(seed)
    D = feat_tr.shape[1]
    W = r.normal(0, np.sqrt(1 / D), (D, n_classes))
    b = np.zeros(n_classes)
    Y = np.eye(n_classes)[ytr]
    eta = 0.5
    for _ in range(epochs):
        p = softmax(feat_tr @ W + b)
        dscore = (p - Y) / len(feat_tr)
        W -= eta * (feat_tr.T @ dscore)
        b -= eta * dscore.sum(0)
    return W, b

print(f"{'target examples':>17}{'transfer (frozen)':>19}{'from scratch':>14}")
rng_sub = np.random.default_rng(35)
for n_per_class in (2, 3, 5, 15, 40, 100):
    idx = []
    for c in range(5):
        class_idx = np.where(ytgt_tr == c)[0]
        idx.extend(rng_sub.choice(class_idx, size=min(n_per_class, len(class_idx)),
                                  replace=False))
    idx = np.array(idx)
    Xsub, ysub = Xtgt_tr[idx], ytgt_tr[idx]

    feat_sub = extract_features(Xsub, src_filters)
    feat_te = extract_features(Xtgt_te, src_filters)
    W, b = train_cnn_head_only(feat_sub, ysub, n_classes=5, seed=35)
    acc_transfer = (softmax(feat_te @ W + b).argmax(1) == ytgt_te).mean()

    _, _, _, acc_scratch = train_cnn(Xsub, ysub, Xtgt_te, ytgt_te,
                                     n_classes=5, seed=35, epochs=150)

    print(f"{len(idx):>17}{acc_transfer:>19.4f}{acc_scratch:>14.4f}")

### Block 4  (`c4.py`)

In [ ]:
# Fine-tuning starts from the transferred filters rather than random
# ones, then keeps training everything, filters included, on the
# target task. It should combine a good starting point with the
# ability to specialize.
def fine_tune(Xtr, ytr, Xte, yte, init_filters, n_classes, seed, epochs=150):
    r = np.random.default_rng(seed)
    filters = init_filters.copy()                    # start from transfer, not random
    D_flat = filters.shape[0] * 3 * 3
    Wf = r.normal(0, np.sqrt(2 / D_flat), (D_flat, n_classes))
    bf = np.zeros(n_classes)
    Y = np.eye(n_classes)[ytr]
    eta = 0.3
    for _ in range(epochs):
        conv_out, windows = conv_forward(Xtr, filters)
        relu_out = np.maximum(0, conv_out)
        pool_out, mask = pool_forward(relu_out)
        flat = pool_out.reshape(len(Xtr), -1)
        p = softmax(flat @ Wf + bf)
        dscore = (p - Y) / len(Xtr)
        dWf = flat.T @ dscore; dbf = dscore.sum(0)
        dflat = dscore @ Wf.T
        dpool = dflat.reshape(pool_out.shape)
        drelu = pool_backward(dpool, mask)
        dconv = drelu * (conv_out > 0)
        _, dfilters = conv_backward(dconv, windows, filters)
        Wf -= eta * dWf; bf -= eta * dbf
        filters -= eta * dfilters
    c_te, _ = conv_forward(Xte, filters)
    p_te, _ = pool_forward(np.maximum(0, c_te))
    pred = softmax(p_te.reshape(len(Xte), -1) @ Wf + bf).argmax(1)
    return (pred == yte).mean()

print(f"{'target examples':>17}{'frozen transfer':>18}{'from scratch':>14}{'fine-tuned':>12}")
rng_sub2 = np.random.default_rng(35)
for n_per_class in (2, 5, 15, 40):
    idx = []
    for c in range(5):
        class_idx = np.where(ytgt_tr == c)[0]
        idx.extend(rng_sub2.choice(class_idx, size=min(n_per_class, len(class_idx)),
                                   replace=False))
    idx = np.array(idx)
    Xsub, ysub = Xtgt_tr[idx], ytgt_tr[idx]

    feat_sub = extract_features(Xsub, src_filters)
    feat_te = extract_features(Xtgt_te, src_filters)
    W, b = train_cnn_head_only(feat_sub, ysub, n_classes=5, seed=35)
    acc_frozen = (softmax(feat_te @ W + b).argmax(1) == ytgt_te).mean()

    _, _, _, acc_scratch = train_cnn(Xsub, ysub, Xtgt_te, ytgt_te,
                                     n_classes=5, seed=35, epochs=150)

    acc_finetune = fine_tune(Xsub, ysub, Xtgt_te, ytgt_te, src_filters,
                             n_classes=5, seed=35)

    print(f"{len(idx):>17}{acc_frozen:>18.4f}{acc_scratch:>14.4f}{acc_finetune:>12.4f}")

### Block 5  (`c5.py`)

In [ ]:
# An embedding is whatever a network's pooled features encode before the
# final classification layer. The transferred filters were never trained
# to distinguish digits five through nine, yet if their features are
# genuinely general, examples of the same target digit should still land
# near each other in that space, with no target-task training at all.
feat_te_embed = extract_features(Xtgt_te, src_filters)         # zero target training

from sklearn.decomposition import PCA
proj = PCA(n_components=2, random_state=35).fit_transform(feat_te_embed)

print(f"embedding dimension (pooled, flattened): {feat_te_embed.shape[1]}")
print(f"reduced to 2 dimensions for inspection, via Chapter 27's PCA")

# nearest-neighbor classification directly in the embedding space --
# no training at all, just distance in the transferred feature space
from sklearn.neighbors import KNeighborsClassifier
feat_tr_embed = extract_features(Xtgt_tr, src_filters)
knn = KNeighborsClassifier(5).fit(feat_tr_embed, ytgt_tr)
knn_acc = knn.score(feat_te_embed, ytgt_te)
print(f"\nk-NN accuracy directly in the transferred embedding space: {knn_acc:.4f}")
print(f"(zero target-task training: no head, no fine-tuning, just distance)")

# compare against k-NN on raw pixels, no embedding at all
knn_raw = KNeighborsClassifier(5).fit(Xtgt_tr.reshape(len(Xtgt_tr), -1), ytgt_tr)
knn_raw_acc = knn_raw.score(Xtgt_te.reshape(len(Xtgt_te), -1), ytgt_te)
print(f"k-NN accuracy on raw pixels, no embedding at all:      {knn_raw_acc:.4f}")